# RAISE-26: Mirror, Mirror on the Wall — Is AI Transforming Us All?(Synaptic Sparks)

**Goal (Competition framing):** Use NLP on AI-related news headlines to study *how AI is shaping the ways humans think, act, and interact*.

This Colab merges the three project notebooks into one coherent workflow:
1) **Dataset A (News headlines)**: multi-label behavior taxonomy + time/source metadata
2) **Modeling**: strong baseline (TF‑IDF + metadata + One-vs-Rest Logistic Regression) + optional Transformer (PyTorch/HF)
3) **Interpretability & Insight**: label quality control, trend analysis, sentiment, topic modeling (NMF)
4) **Dataset C (LLM outputs)**: apply the trained classifier to compare how different LLMs talk about AI’s influence (by persona/role)

> **Deliverable style:** This notebook is written to be readable as a report: each section contains (a) what we do, (b) why it matters for RAISE‑26, and (c) outputs to paste into your final submission.


##Setup & Reproducibility

- Installs required packages (Colab)
- Sets random seeds
- Defines file paths

**Input files**
- `dataset_A_news_full_10500.csv`
- `Dataset_C_prompts_&_queries.csv`
**bold text**

In [ ]:
# --- If your Colab environment is broken (numpy/scipy/sklearn import errors), run this ONCE ---
# Then: Runtime -> Restart runtime, and re-run from the top.
#
# Symptoms this fixes:
# - WARNING: Ignoring invalid distribution ~cipy
# - ImportError: cannot import name '_center' from numpy._core.umath
# - sklearn failing to import because scipy/numpy are inconsistent
#
# Uncomment to run:
# !pip -q uninstall -y numpy scipy scikit-learn pandas
# !pip -q install --no-cache-dir --force-reinstall numpy scipy scikit-learn
# !pip -q install --no-cache-dir --force-reinstall pandas==2.2.2
#
# (If you still see '~cipy', run: !ls /usr/local/lib/python*/dist-packages | grep '~cipy'
#  and remove those folders with rm -rf)


In [ ]:
# --- Colab installs (safe; avoids upgrading core Colab stack) ---
# IMPORTANT: Do NOT upgrade numpy/pandas/scipy/scikit-learn in Colab unless you know why.
# Upgrading them often breaks binary wheels and causes errors like "Ignoring invalid distribution ~cipy".

!pip -q install -U nltk
!pip -q install -U iterative-stratification

# Optional (deep model). Colab usually already has torch; keep it.
!pip -q install -U transformers datasets accelerate

import os, re, json, math, random
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt


In [ ]:
# --- Paths ---
# Colab: if you uploaded the CSVs, they are in /content/
# Local runner (this environment): files are in /mnt/data/

PATH_A_CANDIDATES = [
    '/content/dataset_A_news_full_10500.csv',
    '/mnt/data/dataset_A_news_full_10500.csv'
]
PATH_C_CANDIDATES = [
    '/content/Dataset_C_prompts_&_queries.csv',
    '/content/Dataset_C_prompts_&_queries.csv'.replace(' ', ''),
    '/mnt/data/Dataset_C_prompts_&_queries.csv'
]

def pick_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates}. In Colab, upload the file to /content.")

PATH_A = pick_path(PATH_A_CANDIDATES)
PATH_C = pick_path(PATH_C_CANDIDATES)

TEXT_COL = 'title'
LABEL_COL = 'classes_str'

print('PATH_A:', PATH_A)
print('PATH_C:', PATH_C)


## 1) Load Dataset A (News headlines)

Dataset A already contains:
- headline text (`title`)
- time fields (date, year, month, quarter, day_of_week, is_weekend)
- source
- engineered length features
- **multi-label behavior tags** in `classes_str` (semicolon-separated)

For RAISE‑26, we interpret each label as a *behavioral impact dimension* (e.g., cognition/decision-making, work/economy, society/ethics, etc.).

In [ ]:
dfA = pd.read_csv(PATH_A)
print('Shape:', dfA.shape)
print('Columns:', dfA.columns.tolist())

# Basic clean
dfA = dfA.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

dfA['date'] = pd.to_datetime(dfA['date'], errors='coerce')

# Normalize text
def clean_text(s: str) -> str:
    s = '' if pd.isna(s) else str(s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

dfA[TEXT_COL] = dfA[TEXT_COL].map(clean_text)

print('After clean:', dfA.shape)
dfA.head(3)


> **Analyst Takeaway:**  
> Clean preprocessing ensures unbiased downstream NLP results.To ensure fair and reliable modeling, Dataset A was quality-checked and cleaned by removing incomplete records, standardizing temporal information, and lightly normalizing headline text. This preprocessing step establishes a consistent and unbiased foundation for subsequent NLP analysis.


## 2) Parse Multi-Label Targets + Label Quality Control

Here we:
- split semicolon-separated labels into a list
- compute frequency of each label
- optionally merge very rare labels into `OTHER_RARE` (report your choice)

> Tip: In your final report, include the frequency plot + your rare-label decision. That is a strong “label QC” section.

In [ ]:
# Split the semicolon-separated labels into a list

def parse_labels(s: str):
    parts = [p.strip() for p in str(s).split(';')]
    parts = [p for p in parts if p]
    return parts

dfA['labels_list'] = dfA[LABEL_COL].map(parse_labels)

# Label universe
all_labels = sorted({lab for labs in dfA['labels_list'] for lab in labs})
print('Num base labels:', len(all_labels))

# Frequency
lab_counts = pd.Series([lab for labs in dfA['labels_list'] for lab in labs]).value_counts()
lab_counts.head(15)


> **Analyst Takeaway:**  
> Label frequency analysis shows a well-balanced distribution across 12 core categories, with no extreme long-tail sparsity. All labels retain sufficient sample support, and therefore no labels were merged or removed. The full label set was preserved for modeling to maintain interpretability and coverage of human-centered AI impact themes.

In [ ]:
# Plot top labels
plt.figure(figsize=(10,4))
lab_counts.head(15).plot(kind='bar')
plt.title('Top 15 behavior labels in Dataset A')
plt.ylabel('Count')
plt.show()

print('Total unique behavior labels:', len(lab_counts))
print('Rare labels (count<=10):', int((lab_counts<=10).sum()))


> **Analyst Takeaway:**  
> The label frequency plot indicates a balanced distribution across all 12 behavior categories, with no rare labels (count ≤ 10) observed. As a result, no label merging or exclusion was required, and the full label set was retained for modeling.

In [ ]:
# Optional: merge very rare labels to reduce label sparsity
# You can set RARE_MIN_COUNT higher (e.g., 20) if you want more stability.
RARE_MIN_COUNT = 10
rare = set(lab_counts[lab_counts < RARE_MIN_COUNT].index)

MERGE_RARE = True

def merge_rare_labels(labs):
    if not MERGE_RARE:
        return labs
    kept = [l for l in labs if l not in rare]
    if len(kept) < len(labs):
        kept.append('OTHER_RARE')
    return sorted(set(kept))

dfA['labels_final'] = dfA['labels_list'].map(merge_rare_labels)

all_labels_final = sorted({lab for labs in dfA['labels_final'] for lab in labs})
print('Num labels after merge:', len(all_labels_final))

lab_counts_final = pd.Series([lab for labs in dfA['labels_final'] for lab in labs]).value_counts()
lab_counts_final.head(15)


> **Analyst Takeaway:**  
>An optional rare-label merging procedure was evaluated to mitigate potential label sparsity. Using a minimum frequency threshold of 10, no labels met the rare criteria; therefore, no aggregation was applied, and the original 12-category label structure was preserved.

## 3) Train/Val/Test Split (Multi-label aware)

We use *iterative stratification* (better for multi-label than plain random splits).

Outputs:
- `train_df`, `val_df`, `test_df`
- multi-hot matrices `Y_*` aligned with `label_names`

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

label_names = sorted(lab_counts_final.index)
lab2id = {l:i for i,l in enumerate(label_names)}

def to_multihot(labs):
    y = np.zeros(len(label_names), dtype=np.int8)
    for l in labs:
        if l in lab2id:
            y[lab2id[l]] = 1
    return y

Y = np.vstack(dfA['labels_final'].map(to_multihot).values)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(msss.split(dfA, Y))

train_df = dfA.iloc[train_idx].reset_index(drop=True)
temp_df  = dfA.iloc[temp_idx].reset_index(drop=True)
Y_train = Y[train_idx]
Y_temp  = Y[temp_idx]

msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(msss2.split(temp_df, Y_temp))

val_df  = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

Y_val  = Y_temp[val_idx]
Y_test = Y_temp[test_idx]

print('Train/Val/Test:', len(train_df), len(val_df), len(test_df))


**Analyst Takeaway:**
>A multi-label–aware train/validation/test split was constructed using iterative stratification to preserve label distributions across subsets. The final split (Train: 8,393; Validation: 1,041; Test: 1,066) ensures balanced and representative data partitions for robust model training and evaluation.

>（中文）：
采用多标签感知的迭代分层方法对数据进行训练集、验证集和测试集划分，以保持各标签在不同子集中的分布一致性。最终划分结果（训练集 8,393；验证集 1,041；测试集 1,066）在规模和标签结构上均具备良好代表性，支持稳健的模型训练与评估。

>(한국어):
다중 레이블 분포를 각 데이터 분할에 균등하게 유지하기 위해 반복적 계층화(iterative stratification)를 적용하여 학습/검증/테스트 세트를 구성하였다. 최종 분할 결과(학습 8,393 / 검증 1,041 / 테스트 1,066)는 레이블 편향을 최소화하며, 안정적인 모델 학습 및 평가를 가능하게 한다.

## 4) Strong Baseline: TF‑IDF (title) + Metadata + One-vs-Rest Logistic Regression

**Why this matters:**
- Fast, reliable, easy to explain
- Provides interpretable coefficients (top n-grams per label)

We combine:
- Text features: TF‑IDF n-grams
- Metadata: one-hot (source, day_of_week, month) + numeric scaling (length features, year, quarter, is_weekend)

Model type:
- **Multi-label** OneVsRest Logistic Regression

Evaluation:
- micro-F1, macro-F1
- per-label F1 table

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

# Detect columns
cat_cols = [c for c in ['source','day_of_week','month'] if c in dfA.columns]
num_cols = [c for c in ['number_of_characters_title','number_of_words_title','year','quarter','is_weekend'] if c in dfA.columns]

text_col = TEXT_COL

preprocess = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=60000), text_col),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(with_mean=False), num_cols),
    ],
    remainder='drop'
)

clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        n_jobs=-1,
        class_weight='balanced',
        solver='liblinear'
    )
)

baseline = Pipeline([
    ('prep', preprocess),
    ('clf', clf)
])

baseline


**Analyst TakeAway:**
>A TF-IDF–based One-vs-Rest Logistic Regression baseline was selected to provide a fast, stable, and interpretable reference model for multi-label classification. Combining headline text with lightweight metadata captures both semantic content and contextual signals, while preserving transparency through interpretable feature weights. This baseline establishes a clear and explainable benchmark against which more complex models can be evaluated.

>选择基于 TF-IDF 的 One-vs-Rest 逻辑回归作为基线模型，旨在为多标签任务提供一个快速、稳定且具备可解释性的参考框架。通过将标题文本与轻量级元数据特征相结合，模型能够同时捕捉语义信息与上下文信号，并通过可解释的特征权重保持分析透明度，从而为后续复杂模型的对比评估建立清晰基准。

>다중 레이블 분류를 위한 기준 모델로 TF-IDF 기반 One-vs-Rest 로지스틱 회귀를 선택하였다. 본 접근법은 제목 텍스트와 경량 메타데이터를 결합하여 의미적 정보와 맥락적 신호를 함께 포착하면서도, 해석 가능한 가중치를 통해 분석의 투명성을 유지한다. 이는 보다 복잡한 모델과의 비교를 위한 명확하고 설명 가능한 기준선을 제공한다.

In [ ]:
baseline.fit(train_df, Y_train)

# Predict probabilities (not all LR backends support predict_proba in OvR consistently; use decision_function + sigmoid)
from scipy.special import expit

def predict_proba_ovr(pipe, X):
    scores = pipe.decision_function(X)
    return expit(scores)

proba_val = predict_proba_ovr(baseline, val_df)
pred_val = (proba_val >= 0.5).astype(int)

micro = f1_score(Y_val, pred_val, average='micro', zero_division=0)
macro = f1_score(Y_val, pred_val, average='macro', zero_division=0)
print('Baseline Val micro-F1:', round(micro,4))
print('Baseline Val macro-F1:', round(macro,4))


**warning: need send look**

In [ ]:
# Per-label F1
per_f1 = []
for i,lbl in enumerate(label_names):
    per_f1.append((lbl, f1_score(Y_val[:,i], pred_val[:,i], zero_division=0)))

per_f1_df = pd.DataFrame(per_f1, columns=['label','f1']).sort_values('f1')
per_f1_df.head(15)


**Analyst takeaway:**
>In addition to strong overall validation performance, per-label F1 analysis reveals consistent effectiveness across all behavior categories. Higher F1 scores are observed for structurally explicit themes such as Work, Jobs & Economy, Social Interaction & Relationships, and Technology & Interaction, indicating clear lexical and contextual signals. Relatively lower—but still robust—performance on affective categories such as Emotion, Motivation & Well-being suggests greater semantic nuance and overlap. Overall, the baseline demonstrates balanced multi-label behavior recognition and provides meaningful insight into label-specific difficulty.

>除整体验证集性能表现优异外，逐标签 F1 分析显示模型在各行为类别上均具备稳定识别能力。诸如“工作、经济”“社会互动”“技术与交互”等结构性较强的主题取得更高 F1 分数，反映其语义和上下文信号较为明确；而“情绪、动机与幸福感”等情感类标签虽然表现略低，但仍保持较高水平，体现其语义更为细腻且存在交叉。整体来看，该基线模型在多标签行为识别上表现均衡，并有效揭示了不同标签的相对建模难度。

>전체 검증 성능이 우수한 가운데, 레이블별 F1 분석에서도 모든 행동 범주에 대해 안정적인 성능이 확인되었다. Work, Jobs & Economy, Social Interaction & Relationships, Technology & Interaction과 같이 구조적이고 명시적인 주제는 높은 F1 점수를 보이며, 이는 명확한 어휘 및 맥락 신호를 반영한다. 반면 Emotion, Motivation & Well-being과 같은 정서적 범주는 상대적으로 낮지만 여전히 견고한 성능을 보였으며, 이는 의미적 미묘함과 중첩 특성을 시사한다. 전반적으로 본 베이스라인은 레이블별 난이도를 잘 드러내는 균형 잡힌 다중 레이블 분류 성능을 제공한다.

## 4.1) Multi-label confusion: which labels are over/under-predicted?

For multi-label problems, a single confusion matrix is not well-defined. A practical alternative is to compute per-label **TP/FP/FN/TN** and inspect which labels tend to be:
- **Under-predicted** (high FN)
- **Over-predicted** (high FP)


In [ ]:
# Per-label confusion stats on the validation set

def per_label_confusion(y_true, y_pred, names):
    rows = []
    for i,lab in enumerate(names):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        tp = int(((yt==1) & (yp==1)).sum())
        fp = int(((yt==0) & (yp==1)).sum())
        fn = int(((yt==1) & (yp==0)).sum())
        tn = int(((yt==0) & (yp==0)).sum())
        rows.append((lab,tp,fp,fn,tn))
    out = pd.DataFrame(rows, columns=['label','TP','FP','FN','TN'])
    out['precision'] = out['TP'] / (out['TP'] + out['FP'] + 1e-9)
    out['recall'] = out['TP'] / (out['TP'] + out['FN'] + 1e-9)
    out['f1'] = 2*out['precision']*out['recall'] / (out['precision']+out['recall'] + 1e-9)
    return out

conf_val = per_label_confusion(Y_val, pred_val, label_names)

print('Most under-predicted (highest FN):')
display(conf_val.sort_values('FN', ascending=False).head(8))

print('Most over-predicted (highest FP):')
display(conf_val.sort_values('FP', ascending=False).head(8))


**Analyst takeaway:**
> A per-label confusion analysis highlights systematic differences in prediction behavior across categories. Labels such as Sentiment (Positive / Negative Feelings) and Emotion, Motivation & Well-being exhibit higher false negatives, indicating under-prediction driven by semantic subtlety and overlap with other behavior types. In contrast, categories including Cognitive & Decision-Making and Society, Ethics & Culture show relatively higher false positives, suggesting broader conceptual boundaries that invite over-assignment. These patterns are consistent with the abstract or affective nature of certain labels and provide guidance for targeted model refinement.

>逐标签混淆分析揭示了不同类别在预测行为上的系统性差异。“情绪（正/负）”与“情感、动机与幸福感”等标签存在较高的漏判（FN），表明其语义细腻、与其他行为类型存在重叠，模型更倾向于保守预测；而“认知与决策”“社会、伦理与文化”等类别则表现出较高的误判（FP），反映其概念边界相对宽泛，容易被过度分配。这些误差模式与标签的抽象性和情感属性高度一致，并为后续模型改进提供了明确方向。

>레이블별 혼동 분석 결과, 범주에 따라 예측 편향이 다르게 나타났다. Sentiment (Positive / Negative Feelings) 및 Emotion, Motivation & Well-being과 같은 정서적 레이블은 높은 FN 비율을 보이며, 이는 의미적 미묘함과 레이블 간 중첩으로 인한 과소 예측 경향을 시사한다. 반면 Cognitive & Decision-Making, Society, Ethics & Culture는 상대적으로 FP가 높아 개념적 범위가 넓고 과대 예측되기 쉬운 특성을 보인다. 이러한 패턴은 레이블의 추상적·정서적 성격과 일관되며, 향후 모델 개선을 위한 중요한 진단 근거를 제공한다.

## 4.2) Save + Inference helpers

Saving the baseline pipeline makes it easy to:
- Reuse the classifier for Dataset C
- Deploy a simple demo (paste a headline → top predicted behavior impacts)


In [ ]:
import joblib

# Helper to create a feature frame matching Dataset A's schema
# (works for single headline inference and Dataset C outputs)

def build_feature_frame(text_series: pd.Series) -> pd.DataFrame:
    text_series = text_series.fillna('').astype(str)
    df = pd.DataFrame({TEXT_COL: text_series})

    # Categorical placeholders
    for c in cat_cols:
        df[c] = 'UNKNOWN'

    # Numeric proxies
    if 'number_of_characters_title' in num_cols:
        df['number_of_characters_title'] = text_series.str.len()
    if 'number_of_words_title' in num_cols:
        df['number_of_words_title'] = text_series.str.split().map(len)

    # Time-like defaults (if needed)
    if 'year' in num_cols:
        df['year'] = 0
    if 'quarter' in num_cols:
        df['quarter'] = 0
    if 'is_weekend' in num_cols:
        df['is_weekend'] = False

    # Ensure all expected numeric cols exist
    for n in num_cols:
        if n not in df.columns:
            df[n] = 0

    return df

MODEL_OUT = 'raise26_baseline.joblib'
joblib.dump({'pipeline': baseline, 'label_names': label_names, 'cat_cols': cat_cols, 'num_cols': num_cols}, MODEL_OUT)
print('Saved:', MODEL_OUT)

@np.errstate(over='ignore')
def predict_top_k(headline: str, k: int = 5):
    x = build_feature_frame(pd.Series([headline]))
    p = predict_proba_ovr(baseline, x)[0]
    top = p.argsort()[::-1][:k]
    return [(label_names[i], float(p[i])) for i in top]

sample = "AI assistant introduced at work to summarize meetings"
print('Sample:', sample)
for lab,prob in predict_top_k(sample, k=5):
    print(f"{lab}: {prob:.3f}")


**Analyst Takeaway**
> The trained baseline pipeline was serialized to enable reproducible reuse for downstream inference, including applying the classifier to Dataset C and supporting lightweight demo deployment. A schema-aligned feature builder was implemented to ensure inference inputs remain compatible with the training pipeline even when metadata is missing, using controlled placeholders and simple text-derived proxies. A top-k prediction helper provides interpretable, user-facing outputs, and a sample headline yields plausible high-confidence labels, serving as a practical sanity check.

>为支持可复现的下游推理与复用，将训练完成的基线 pipeline 进行序列化保存，便于直接应用于 Dataset C 及简易演示场景。针对推理阶段可能缺失元数据字段的问题，实现了与训练 schema 对齐的特征构造函数，通过受控占位符与基于文本的简单代理特征保证输入兼容性。Top-k 预测接口输出可解释的高概率标签，样例标题得到与语义一致的高置信预测结果，可作为有效的合理性检验。

>학습된 베이스라인 파이프라인을 직렬화하여 재현 가능한 재사용을 가능하게 했으며, Dataset C 적용 및 간단한 데모 배포에 활용할 수 있도록 구성하였다. 추론 시 메타데이터가 없더라도 학습 스키마와 호환되도록 placeholder와 텍스트 기반 proxy 특징을 사용하는 입력 생성 함수를 구현하였다. 또한 top-k 예측 헬퍼를 통해 해석 가능한 사용자 친화적 결과를 제공하며, 예시 헤드라인에서도 의미적으로 타당한 고신뢰 레이블이 출력되어 실용적인 sanity check를 수행했다.

## 5) Interpretability: Top n-grams per Label

We extract high-weight n-grams from each label’s logistic regression classifier.
This helps turn “model output” into **human-readable insight** for RAISE‑26.

In [ ]:
# Utility to pull feature names from ColumnTransformer

def get_feature_names(ct: ColumnTransformer):
    names = []
    for name, trans, cols in ct.transformers_:
        if name == 'remainder':
            continue
        if hasattr(trans, 'get_feature_names_out'):
            if name == 'text':
                fn = trans.get_feature_names_out()
            else:
                fn = trans.get_feature_names_out(cols)
            names.extend([f'{name}__{x}' for x in fn])
        else:
            # fallback
            if isinstance(cols, list):
                names.extend([f'{name}__{c}' for c in cols])
            else:
                names.append(f'{name}__{cols}')
    return np.array(names)

feat_names = get_feature_names(baseline.named_steps['prep'])

# Each OvR estimator corresponds to a label
estimators = baseline.named_steps['clf'].estimators_

def top_ngrams_for_label(label_idx, top_n=15, only_text=True):
    est = estimators[label_idx]
    coefs = est.coef_.ravel()
    if only_text:
        mask = np.array([n.startswith('text__') for n in feat_names])
        coefs2 = coefs[mask]
        feats2 = feat_names[mask]
    else:
        coefs2 = coefs
        feats2 = feat_names
    top_pos = np.argsort(coefs2)[::-1][:top_n]
    top_neg = np.argsort(coefs2)[:top_n]
    return pd.DataFrame({
        'top_positive': feats2[top_pos],
        'w_pos': coefs2[top_pos],
        'top_negative': feats2[top_neg],
        'w_neg': coefs2[top_neg],
    })

# Example: show interpretability for the most frequent label
most_common_label = lab_counts_final.index[0]
idx = label_names.index(most_common_label)
print('Most common label:', most_common_label)
top_ngrams_for_label(idx, top_n=12)


**Analyst Takeaway:**
>To translate baseline predictions into human-readable evidence, we extracted the highest-weight TF-IDF n-grams from each label’s One-vs-Rest logistic regression classifier. For the most frequent category (Work, Jobs & Economy), top positive n-grams emphasize workplace and productivity concepts (e.g., work, industry, management, automation), aligning with the intended behavioral theme. Conversely, high-magnitude negative n-grams cluster around learning/education and general technology terms, indicating how the model distinguishes adjacent narratives and reduces cross-label confusion. This interpretability step strengthens the credibility of findings by linking label assignments to transparent lexical signals.

>为将基线模型输出转化为可直接解读的证据，我们从每个标签对应的 One-vs-Rest 逻辑回归分类器中提取高权重的 TF-IDF n-grams。以最常见的“工作、就业与经济”为例，正向高权重词集中在工作场景与效率/自动化等概念（如 work、industry、management、automation），与标签语义高度一致；负向高权重词则更多聚集在学习/教育及更泛化的技术叙事上，体现模型用于区分相邻主题、降低跨标签混淆的机制。该可解释性分析为比赛报告提供了可追溯的文本证据支撑。

베이스라인 예측을 사람이 이해할 수 있는 근거로 연결하기 위해, 각 레이블의 One-vs-Rest 로지스틱 회귀 분류기에서 가중치가 큰 TF-IDF n-gram을 추출하였다. 가장 빈도가 높은 Work, Jobs & Economy 레이블의 경우, 긍정 가중치는 업무·생산성·자동화와 관련된 표현(work, industry, management, automation 등)에 집중되어 레이블 의미와 일치한다. 반대로 부정 가중치는 학습/교육 및 일반 기술 서사에 더 많이 분포하여, 인접한 주제 간을 구분하고 레이블 혼동을 줄이는 모델의 분별 기준을 보여준다. 이 단계는 레이블 부여를 투명한 텍스트 신호와 연결함으로써 결과의 신뢰도를 강화한다.

## 6) Error Analysis (High-confidence mistakes)

Judges like seeing you *diagnose failure modes* and link them back to behavior/social narratives.

We create a table of cases where the model is very confident but wrong for a chosen label.

In [ ]:
# Build an error table for a chosen label
LABEL_TO_INSPECT = most_common_label
j = label_names.index(LABEL_TO_INSPECT)

val_out = val_df[[TEXT_COL, 'source', 'date', LABEL_COL]].copy()
val_out['true'] = Y_val[:,j]
val_out['proba'] = proba_val[:,j]
val_out['pred'] = (val_out['proba'] >= 0.5).astype(int)
val_out['correct'] = (val_out['true'] == val_out['pred'])

# High-confidence errors
high_conf_errors = val_out[(~val_out['correct']) & ((val_out['proba'] >= 0.9) | (val_out['proba'] <= 0.1))]
high_conf_errors.sort_values('proba', ascending=False).head(20)


**Analyst Takeaway:**
>To diagnose model failure modes beyond aggregate metrics, we examined high-confidence misclassifications for the most frequent label (Work, Jobs & Economy). These cases represent instances where the model assigns extreme probabilities yet produces incorrect predictions, revealing systematic ambiguity in how AI-related narratives overlap across behavioral categories. Inspection of such errors suggests that headlines blending workplace themes with education, technology, or social framing can trigger confident but incorrect label assignments, highlighting the limits of surface-level lexical cues. This analysis underscores the importance of contextual modeling when interpreting AI’s behavioral and social impacts.

>为超越整体指标层面的评估，我们针对最常见的“工作、就业与经济”标签，分析了模型在高置信度情况下仍然预测错误的样本。这类错误揭示了 AI 相关新闻叙事在不同行为类别之间的系统性重叠：当标题同时涉及工作场景与教育、技术或社会语境时，模型容易基于显著词汇做出高置信但不完全准确的判断。该结果反映了仅依赖表层词汇信号的局限性，并强调在分析 AI 行为与社会影响时引入更强上下文建模的重要性。

>전체 성능 지표를 넘어 모델의 실패 양상을 진단하기 위해, 가장 빈도가 높은 Work, Jobs & Economy 레이블에 대해 고신뢰도 오분류 사례를 분석하였다. 이러한 사례는 모델이 매우 높은 확률을 부여했음에도 잘못된 예측을 한 경우로, AI 관련 서사가 여러 행동 범주에 걸쳐 중첩되는 구조적 모호성을 드러낸다. 특히 업무 맥락과 교육, 기술, 사회적 프레이밍이 혼합된 헤드라인에서 표면적 어휘 단서에 기반한 과신 오류가 발생함을 확인하였다. 이는 AI의 행동적·사회적 영향을 해석하는 데 있어 보다 맥락 중심적인 모델의 필요성을 시사한다.

## 7) Insight Layer A: Trend Analysis over Time & Source

This directly answers the competition question: **what behavior domains does the AI news ecosystem emphasize, and how does that change over time?**

We compute per-quarter label prevalence and show simple trend plots.

In [ ]:
# Label prevalence by quarter
work_df = dfA.copy()

# Ensure quarter exists; else derive
if 'quarter' not in work_df.columns and 'date' in work_df.columns:
    work_df['quarter'] = work_df['date'].dt.quarter
if 'year' not in work_df.columns and 'date' in work_df.columns:
    work_df['year'] = work_df['date'].dt.year

# Multi-hot per row (final labels)
Y_all = np.vstack(work_df['labels_final'].map(to_multihot).values)

work_df['year_quarter'] = work_df['year'].astype(str) + '-Q' + work_df['quarter'].astype(str)

# prevalence = mean of multi-hot per time bucket
prev = pd.DataFrame(Y_all, columns=label_names)
prev['year_quarter'] = work_df['year_quarter'].values
prev_q = prev.groupby('year_quarter')[label_names].mean().sort_index()

prev_q.tail(5)


**Analyst Takeaway:**

>Quarterly label prevalence analysis reveals stable yet differentiated emphasis across behavioral domains in AI-related news. Work, Jobs & Economy consistently dominates coverage, underscoring the centrality of labor and productivity narratives in discussions of AI impact. Categories such as Technology & Interaction and Learning, Knowledge & Education remain persistently salient, while affective and relational domains (e.g., Emotion, Motivation & Well-being, Social Interaction & Relationships) appear at lower but steady levels. Minor quarter-to-quarter variation suggests that the AI news ecosystem prioritizes structural and economic implications over short-term emotional framing, indicating sustained narrative focus rather than episodic fluctuation.

>按季度计算的标签占比显示，AI 相关新闻在不同行为领域上的关注重点整体稳定但层次分明。“工作、就业与经济”长期占据主导地位，反映出 AI 影响讨论中对劳动、生产力与组织结构的持续关注；“技术与交互”“学习、知识与教育”等类别同样保持较高且稳定的出现频率。相比之下，情绪与社会关系类行为维度虽占比较低，但随时间保持平稳。整体趋势表明，AI 新闻生态更强调结构性与经济性影响，而非短期情绪波动，呈现出持续而非事件驱动的叙事模式。

>분기별 레이블 출현 비율 분석 결과, AI 관련 뉴스는 행동 영역별로 안정적이면서도 차별화된 강조 양상을 보인다. Work, Jobs & Economy는 지속적으로 가장 높은 비중을 차지하며, 이는 AI 영향 논의에서 노동과 생산성 서사가 핵심임을 보여준다. Technology & Interaction, Learning, Knowledge & Education 역시 일관되게 높은 중요도를 유지하는 반면, 정서적·관계적 영역은 상대적으로 낮지만 안정적인 수준을 나타낸다. 분기 간 변화 폭이 크지 않다는 점은 AI 뉴스 담론이 단기 이슈보다는 구조적·장기적 영향에 초점을 두고 있음을 시사한다.

In [ ]:
# Plot top 6 labels trend
top6 = lab_counts_final.head(6).index.tolist()
plt.figure(figsize=(12,4))
for l in top6:
    plt.plot(prev_q.index, prev_q[l], label=l)
plt.xticks(rotation=60, ha='right')
plt.title('Quarterly prevalence of top behavior labels (Dataset A)')
plt.ylabel('Share of headlines tagged with label')
plt.legend()
plt.tight_layout()
plt.show()


**Analyst Takeaway:**

>The quarterly trend plot of the top behavior labels shows a consistent prioritization of economic and structural themes in AI-related news. Work, Jobs & Economy remains the most prevalent category and exhibits a slight upward trend, reinforcing the narrative focus on labor and productivity impacts. In contrast, Learning, Knowledge & Education shows a modest decline, while Technology & Interaction and Society, Ethics & Culture display gradual increases, suggesting a slow shift toward broader societal framing. Overall, the limited volatility across quarters indicates stable narrative emphasis rather than event-driven spikes.

>前六大行为标签的季度趋势图显示，AI 相关新闻长期聚焦于经济与结构性议题。“工作、就业与经济”始终占据最高比例，并呈现轻微上升趋势，进一步强化了围绕劳动与生产力影响的核心叙事；相比之下，“学习、知识与教育”出现小幅下降，而“技术与交互”“社会、伦理与文化”则缓慢上升，暗示报道视角正逐步拓展至更广泛的社会层面。整体来看，各类别随时间波动有限，反映出稳定而非事件驱动的叙事重点。>

>상위 6개 행동 레이블의 분기별 추이 그래프는 AI 관련 뉴스가 경제적·구조적 주제를 지속적으로 우선시하고 있음을 보여준다. Work, Jobs & Economy는 가장 높은 비중을 유지하며 소폭 상승하여 노동 및 생산성 영향에 대한 담론이 강화되고 있음을 시사한다. 반면 Learning, Knowledge & Education은 완만한 감소를 보이고, Technology & Interaction과 Society, Ethics & Culture는 점진적으로 증가하여 사회적 프레이밍이 확장되고 있음을 나타낸다. 전반적으로 분기 간 변동성이 크지 않아, 단기 이슈보다는 지속적인 담론 구조가 형성되어 있음을 보여준다.

## 8) Insight Layer B: Sentiment Trend (Headlines)

Sentiment is a *proxy* for perceived benefits vs risks of AI.
We use VADER (headline-friendly).

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

work_df['sentiment'] = work_df[TEXT_COL].map(lambda s: sia.polarity_scores(str(s))['compound'])

sent_q = work_df.groupby('year_quarter')['sentiment'].mean().sort_index()

plt.figure(figsize=(12,3))
plt.plot(sent_q.index, sent_q.values)
plt.xticks(rotation=60, ha='right')
plt.title('Average headline sentiment by quarter')
plt.ylabel('VADER compound')
plt.tight_layout()
plt.show()

sent_q.tail(10)


## 9) Insight Layer C: Topic Modeling (NMF)

Topic modeling surfaces the “storylines” that co-occur with behavior impacts.
We use NMF (stable, explainable) on TF‑IDF features.

In [ ]:
from sklearn.decomposition import NMF

# Fit NMF on TF-IDF of titles only (for interpretability)
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=40000, stop_words='english')
X_t = tfidf.fit_transform(work_df[TEXT_COL])

N_TOPICS = 10
nmf = NMF(n_components=N_TOPICS, random_state=SEED, init='nndsvda', max_iter=400)
W = nmf.fit_transform(X_t)
H = nmf.components_

vocab = np.array(tfidf.get_feature_names_out())

def show_topics(H, vocab, top_n=12):
    topics = []
    for k, comp in enumerate(H):
        top = vocab[np.argsort(comp)[::-1][:top_n]]
        topics.append((k, ', '.join(top)))
    return pd.DataFrame(topics, columns=['topic_id','top_terms'])

topics_df = show_topics(H, vocab, top_n=12)
topics_df


In [ ]:
# Topic trend by quarter
work_df['topic_id'] = W.argmax(axis=1)

topic_q = work_df.groupby(['year_quarter','topic_id']).size().unstack(fill_value=0)
topic_q = topic_q.div(topic_q.sum(axis=1), axis=0).sort_index()

plt.figure(figsize=(12,4))
for k in range(min(N_TOPICS,6)):
    plt.plot(topic_q.index, topic_q[k], label=f'Topic {k}')
plt.xticks(rotation=60, ha='right')
plt.title('Topic share by quarter (top 6 topics)')
plt.ylabel('Share of headlines')
plt.legend()
plt.tight_layout()
plt.show()


## 11) Dataset C: Compare LLM Narratives using the Classifier(Distilbert)

Dataset C contains *persona prompts + user queries* and each LLM's output.
We:
- extract persona/role from the prompt
- classify the output into behavior-impact labels (using our baseline model, trained on A)
- compare label distributions across LLMs and personas

**Why this matters:**
This translates open-ended LLM language into the same behavioral framework, enabling a fair comparison of “what each model emphasizes”.

### Report-ready narrative template

Use the figures/tables produced above and fill in the bracketed spots:

- **Observed behavioral emphasis in AI news:** The most frequent impact categories in Dataset A are **[TOP LABELS]**, suggesting public discussion of AI disproportionately focuses on these behavior domains.

- **Temporal dynamics:** Quarterly prevalence plots show **[LABEL]** rises or falls during **[YEAR-Q]**. Interpret these shifts alongside real-world drivers (e.g., regulation debates, major product launches, layoffs, elections) without overstating causality.

- **Sentiment:** Sentiment-by-quarter indicates the overall tone becomes **[MORE POSITIVE / MORE NEGATIVE / STABLE]** in **[PERIOD]**; cross-check with label prevalence (e.g., spikes in *Health, Safety & Risk* or *Society, Ethics & Culture* often align with more negative tone).

- **Topics:** NMF topics reveal recurring narratives such as **[TOPICS]**. Combine topic words with example headlines to keep interpretation grounded.

- **LLM narrative differences (Dataset C):** When labeling LLM outputs with the classifier, **[LLM A]** emphasizes **[LABELS]** more than **[LLM B]**. Role/persona heatmaps suggest that personas (e.g., *Research Assistant* vs *Ethicist*) systematically shift which behavior domains are foregrounded.

- **So, is AI transforming us all?** Evidence from headlines suggests AI is most visibly reshaping **[WORK / SOCIAL / COGNITIVE / ETHICS / HEALTH]** behaviors, with uneven attention across domains and clear temporal waves. Your closing paragraph should tie (1) the strongest quantitative signals, (2) a small set of representative headlines, and (3) one equity/ethics implication.


##模型架构
# Input Text
#     ↓
# DistilBERT Tokenizer (max_length=128)
#     ↓
# DistilBERT Encoder (768-dim [CLS] embedding)
#     ↓
# Linear Layer (768 → 256)
#     ↓
# ReLU + Dropout (p=0.3)
#     ↓
# Linear Layer (256 → n_labels)
#     ↓
# Sigmoid → Multi-label Predictions
# - Sanh et al., 2019: "DistilBERT, a distilled version of BERT"
# - Tsoumakas & Katakis, 2007: "Multi-Label Classification: An Overview"

In [ ]:
# 安装依赖（首次运行）
!pip install -q torch transformers scikit-learn pandas numpy matplotlib seaborn
!pip install -q iterative-stratification  # 多标签分层采样
print("✓ 依赖安装完成")

In [ ]:
import os
import re
import warnings
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertTokenizer,
    DistilBertModel,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
# 配置
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# 可复现性设置
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# 设备检测
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*60)
print("DistilBERT Multi-Label Classifier")
print("="*60)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ 未检测到GPU，建议: Runtime → Change runtime type → GPU")
print("="*60)

PyTorch Dataset 类

In [ ]:
class MultiLabelDataset(Dataset):
    """
    多标签文本分类数据集

    功能：
    - 自动tokenization
    - 支持训练/推理两种模式
    - 返回BERT所需的input_ids和attention_mask
    """

    def __init__(
        self,
        texts: List[str],
        labels: Optional[np.ndarray] = None,
        tokenizer: DistilBertTokenizer = None,
        max_length: int = 128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.labels is not None:
            item['labels'] = torch.FloatTensor(self.labels[idx])

        return item

print("✓ Dataset类定义完成")

DistilBERT 多标签分类器

In [ ]:
class DistilBertMultiLabelClassifier(nn.Module):
    """
    DistilBERT多标签分类模型

    架构:
        DistilBERT [CLS] (768-dim)
        → Linear (768 → hidden_dim)
        → ReLU
        → Dropout
        → Linear (hidden_dim → n_labels)

    输出原始logits，需要通过sigmoid获得概率
    """

    def __init__(
        self,
        n_labels: int,
        dropout_rate: float = 0.3,
        hidden_dim: int = 256,
        freeze_bert: bool = False
    ):
        super().__init__()

        # 加载预训练DistilBERT
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        # 可选：冻结BERT层
        if freeze_bert:
            for param in self.distilbert.parameters():
                param.requires_grad = False

        # 分类头
        self.pre_classifier = nn.Linear(768, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_dim, n_labels)

        # 保存配置
        self.n_labels = n_labels
        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout_rate

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor
    ) -> torch.Tensor:
        # DistilBERT编码
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 使用[CLS] token表示
        hidden_state = outputs.last_hidden_state[:, 0]

        # 分类头
        x = self.pre_classifier(hidden_state)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        return logits

    def get_config(self) -> Dict:
        return {
            'n_labels': self.n_labels,
            'hidden_dim': self.hidden_dim,
            'dropout_rate': self.dropout_rate
        }

print("✓ 模型架构定义完成")

MultiLabelTrainer 训练器

In [ ]:
class MultiLabelTrainer:
    """
    多标签分类训练器

    特性：
    - BCEWithLogitsLoss损失函数
    - 线性warmup学习率调度
    - 每轮验证与F1指标
    - 早停机制
    - 自动保存最佳模型
    """

    def __init__(
        self,
        model: DistilBertMultiLabelClassifier,
        train_loader: DataLoader,
        val_loader: DataLoader,
        label_names: List[str],
        learning_rate: float = 2e-5,
        num_epochs: int = 5,
        warmup_steps: int = 0,
        device: torch.device = DEVICE
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.label_names = label_names
        self.learning_rate = learning_rate
        self.num_epochs = num_epochs
        self.warmup_steps = warmup_steps
        self.device = device

        # 损失函数
        self.criterion = nn.BCEWithLogitsLoss()

        # 优化器
        self.optimizer = AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.01
        )

        # 学习率调度器
        total_steps = len(train_loader) * num_epochs
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

        # 训练历史
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_micro_f1': [],
            'val_macro_f1': []
        }

        # 最佳模型状态
        self.best_macro_f1 = 0.0
        self.best_model_state = None

        # 验证预测存储
        self._val_preds = None
        self._val_labels = None

    def train_epoch(self) -> float:
        """执行一个训练epoch"""
        self.model.train()
        total_loss = 0.0
        n_batches = len(self.train_loader)

        for batch_idx, batch in enumerate(self.train_loader):
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels = batch['labels'].to(self.device)

            self.optimizer.zero_grad()
            logits = self.model(input_ids, attention_mask)
            loss = self.criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()

            if (batch_idx + 1) % 50 == 0:
                print(f"  Batch {batch_idx+1}/{n_batches} | Loss: {total_loss/(batch_idx+1):.4f}")

        return total_loss / n_batches

    def validate(self) -> Tuple[float, float, float]:
        """验证并返回 (loss, micro_f1, macro_f1)"""
        self.model.eval()
        total_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()

                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()

                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)

        self._val_preds = all_preds
        self._val_labels = all_labels

        avg_loss = total_loss / len(self.val_loader)
        micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
        macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

        return avg_loss, micro_f1, macro_f1

    def train(
        self,
        save_path: str = 'distilbert_multilabel.pth',
        early_stopping_patience: int = 3
    ) -> Dict:
        """完整训练循环"""
        print(f"\n{'='*60}")
        print("开始训练")
        print(f"{'='*60}")
        print(f"Epochs: {self.num_epochs}")
        print(f"Learning Rate: {self.learning_rate}")
        print(f"Device: {self.device}")
        print(f"Train batches: {len(self.train_loader)}")
        print(f"Val batches: {len(self.val_loader)}")
        print(f"{'='*60}\n")

        patience_counter = 0

        for epoch in range(self.num_epochs):
            epoch_start = datetime.now()
            print(f"Epoch {epoch+1}/{self.num_epochs}")
            print("-" * 40)

            train_loss = self.train_epoch()
            val_loss, micro_f1, macro_f1 = self.validate()

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_micro_f1'].append(micro_f1)
            self.history['val_macro_f1'].append(macro_f1)

            epoch_time = (datetime.now() - epoch_start).seconds
            print(f"\n  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss:   {val_loss:.4f}")
            print(f"  Micro-F1:   {micro_f1:.4f}")
            print(f"  Macro-F1:   {macro_f1:.4f}")
            print(f"  Time:       {epoch_time}s")

            if macro_f1 > self.best_macro_f1:
                self.best_macro_f1 = macro_f1
                self.best_model_state = self.model.state_dict().copy()
                patience_counter = 0
                print(f"  ✓ 新的最佳模型! (Macro-F1: {macro_f1:.4f})")
            else:
                patience_counter += 1
                print(f"  无改进 ({patience_counter}/{early_stopping_patience})")

            if patience_counter >= early_stopping_patience:
                print(f"\n⚠ 早停于epoch {epoch+1}")
                break

            print()

        if save_path and self.best_model_state is not None:
            self._save_model(save_path)
            print(f"\n✓ 最佳模型已保存至: {save_path}")

        print(f"\n{'='*60}")
        print("训练完成!")
        print(f"最佳 Macro-F1: {self.best_macro_f1:.4f}")
        print(f"{'='*60}")

        return self.history

    def _save_model(self, path: str):
        checkpoint = {
            'model_state_dict': self.best_model_state,
            'model_config': self.model.get_config(),
            'label_names': self.label_names,
            'best_macro_f1': self.best_macro_f1,
            'history': self.history
        }
        torch.save(checkpoint, path)

    def get_per_label_metrics(self) -> pd.DataFrame:
        """获取每个标签的性能指标"""
        if self._val_preds is None:
            raise ValueError("请先运行validate()")

        rows = []
        for i, label in enumerate(self.label_names):
            y_true = self._val_labels[:, i]
            y_pred = self._val_preds[:, i]

            tp = int(((y_true == 1) & (y_pred == 1)).sum())
            fp = int(((y_true == 0) & (y_pred == 1)).sum())
            fn = int(((y_true == 1) & (y_pred == 0)).sum())
            support = int(y_true.sum())

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

            rows.append({
                'label': label,
                'precision': round(precision, 4),
                'recall': round(recall, 4),
                'f1': round(f1, 4),
                'support': support
            })

        return pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)

print("✓ Trainer类定义完成")

DistilBertPredictor 预测器

In [ ]:
class DistilBertPredictor:
    """
    训练好的模型的推理类

    功能：
    - 加载保存的模型
    - 批量预测
    - DataFrame集成
    """

    def __init__(
        self,
        model_path: str,
        device: torch.device = DEVICE
    ):
        self.device = device

        print(f"加载模型: {model_path}")
        checkpoint = torch.load(model_path, map_location=device)

        self.label_names = checkpoint['label_names']
        model_config = checkpoint['model_config']

        self.model = DistilBertMultiLabelClassifier(
            n_labels=model_config['n_labels'],
            dropout_rate=model_config['dropout_rate'],
            hidden_dim=model_config['hidden_dim']
        )
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(device)
        self.model.eval()

        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

        print(f"✓ 模型加载成功!")
        print(f"  标签数: {len(self.label_names)}")
        print(f"  最佳Macro-F1: {checkpoint.get('best_macro_f1', 'N/A')}")

    def predict(
        self,
        texts: Union[str, List[str]],
        threshold: float = 0.5,
        top_k: Optional[int] = None,
        return_probs: bool = False,
        batch_size: int = 32
    ) -> List[Dict]:
        """
        预测文本标签

        Args:
            texts: 单个或多个文本
            threshold: 概率阈值
            top_k: 返回概率最高的k个标签
            return_probs: 是否返回概率值
            batch_size: 批处理大小
        """
        if isinstance(texts, str):
            texts = [texts]

        results = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            batch_results = self._predict_batch(batch_texts, threshold, top_k, return_probs)
            results.extend(batch_results)

        return results

    def _predict_batch(self, texts, threshold, top_k, return_probs):
        encodings = self.tokenizer(
            texts,
            add_special_tokens=True,
            max_length=128,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encodings['input_ids'].to(self.device)
        attention_mask = encodings['attention_mask'].to(self.device)

        with torch.no_grad():
            logits = self.model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()

        results = []
        for idx, text in enumerate(texts):
            text_probs = probs[idx]

            if top_k is not None:
                top_indices = np.argsort(text_probs)[::-1][:top_k]
                predictions = [(self.label_names[j], float(text_probs[j])) for j in top_indices]
            else:
                predictions = [
                    (self.label_names[j], float(text_probs[j]))
                    for j in range(len(self.label_names))
                    if text_probs[j] >= threshold
                ]
                predictions.sort(key=lambda x: x[1], reverse=True)

            result = {
                'text': text,
                'predicted_labels': [p[0] for p in predictions],
                'predictions': predictions if return_probs else [p[0] for p in predictions]
            }
            results.append(result)

        return results

    def predict_dataframe(
        self,
        df: pd.DataFrame,
        text_col: str,
        threshold: float = 0.5,
        top_k: Optional[int] = None,
        batch_size: int = 32
    ) -> pd.DataFrame:
        """对DataFrame中的文本列进行预测"""
        texts = df[text_col].astype(str).tolist()

        print(f"预测 {len(texts)} 条文本...")
        results = self.predict(texts, threshold, top_k, batch_size=batch_size)

        df = df.copy()
        df['predicted_labels'] = [r['predicted_labels'] for r in results]

        return df

print("✓ Predictor类定义完成")

数据加载与预处理

In [ ]:
def load_and_prepare_data(
    csv_path: str,
    text_col: str = 'title',
    label_col: str = 'classes_str',
    test_size: float = 0.2,
    rare_threshold: int = 10,
    label_separator: str = ';'
) -> Dict:
    """
    加载并预处理数据

    Args:
        csv_path: CSV文件路径
        text_col: 文本列名
        label_col: 标签列名（分号分隔）
        test_size: 验证集比例
        rare_threshold: 稀有标签阈值
        label_separator: 标签分隔符
    """
    print(f"加载数据: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"样本数: {len(df)}")

    # 清洗
    df = df.dropna(subset=[text_col, label_col]).copy()
    df[text_col] = df[text_col].astype(str).str.strip()

    # 解析标签
    def parse_labels(s):
        parts = [p.strip() for p in str(s).split(label_separator)]
        return [p for p in parts if p]

    df['labels_list'] = df[label_col].map(parse_labels)

    # 统计标签
    all_labels = [lab for labs in df['labels_list'] for lab in labs]
    label_counts = pd.Series(all_labels).value_counts()
    print(f"\n原始标签数: {len(label_counts)}")

    # 处理稀有标签
    rare_labels = set(label_counts[label_counts < rare_threshold].index)
    if rare_labels:
        print(f"合并 {len(rare_labels)} 个稀有标签 (< {rare_threshold} 样本)")

        def merge_rare(labs):
            kept = [l for l in labs if l not in rare_labels]
            if len(kept) < len(labs):
                kept.append('OTHER_RARE')
            return sorted(set(kept))

        df['labels_final'] = df['labels_list'].map(merge_rare)
    else:
        df['labels_final'] = df['labels_list']

    # 最终标签集
    label_names = sorted({lab for labs in df['labels_final'] for lab in labs})
    lab2id = {l: i for i, l in enumerate(label_names)}
    print(f"最终标签数: {len(label_names)}")

    # 多热编码
    def to_multihot(labs):
        y = np.zeros(len(label_names), dtype=np.int8)
        for l in labs:
            if l in lab2id:
                y[lab2id[l]] = 1
        return y

    Y = np.vstack(df['labels_final'].map(to_multihot).values)

    # 数据分割
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
        msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
        train_idx, val_idx = next(msss.split(df, Y))
        print("使用多标签分层采样")
    except ImportError:
        print("使用简单随机分割")
        train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=test_size, random_state=SEED)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    Y_train = Y[train_idx]
    Y_val = Y[val_idx]

    print(f"\n训练集: {len(train_df)} | 验证集: {len(val_df)}")

    return {
        'train_df': train_df,
        'val_df': val_df,
        'Y_train': Y_train,
        'Y_val': Y_val,
        'label_names': label_names,
        'lab2id': lab2id,
        'text_col': text_col
    }


def create_dataloaders(
    data: Dict,
    tokenizer: DistilBertTokenizer,
    batch_size: int = 16,
    max_length: int = 128
) -> Tuple[DataLoader, DataLoader]:
    """创建数据加载器"""
    text_col = data['text_col']

    train_texts = data['train_df'][text_col].astype(str).tolist()
    val_texts = data['val_df'][text_col].astype(str).tolist()

    train_ds = MultiLabelDataset(train_texts, data['Y_train'], tokenizer, max_length)
    val_ds = MultiLabelDataset(val_texts, data['Y_val'], tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, num_workers=0)

    return train_loader, val_loader

print("✓ 数据处理函数定义完成")

可视化工具

In [ ]:
def plot_training_history(history: Dict, save_path: Optional[str] = None):
    """绘制训练曲线"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss曲线
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
    axes[0].plot(epochs, history['val_loss'], 'r-s', label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # F1曲线
    axes[1].plot(epochs, history['val_micro_f1'], 'g-o', label='Micro-F1')
    axes[1].plot(epochs, history['val_macro_f1'], 'orange', marker='s', label='Macro-F1')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_title('Validation F1 Scores')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()


def plot_label_distribution(label_counts: pd.Series, title: str = "Label Distribution"):
    """绘制标签分布"""
    plt.figure(figsize=(12, max(6, len(label_counts) * 0.3)))
    label_counts.plot(kind='barh', color='steelblue')
    plt.xlabel('Count')
    plt.title(title)
    plt.tight_layout()
    plt.show()

print("✓ 可视化函数定义完成")

# %% [markdown]
# ## Cell 9: 完整训练流程

# %%
# ============================================================================
# CELL 9: 主训练函数
# ============================================================================

def train_distilbert_model(
    data: Dict,
    batch_size: int = 16,
    learning_rate: float = 2e-5,
    num_epochs: int = 5,
    max_length: int = 128,
    warmup_steps: int = 0,
    dropout_rate: float = 0.3,
    hidden_dim: int = 256,
    save_path: str = 'distilbert_multilabel.pth',
    device: torch.device = DEVICE
) -> MultiLabelTrainer:
    """
    完整训练流程

    Args:
        data: load_and_prepare_data()返回的字典
        batch_size: 批大小 (GPU内存不足时降至8)
        learning_rate: 学习率
        num_epochs: 训练轮数
        max_length: 最大序列长度 (可降至64加速)
        warmup_steps: 学习率warmup步数
        dropout_rate: Dropout概率
        hidden_dim: 隐藏层维度
        save_path: 模型保存路径
        device: 计算设备
    """
    # 初始化tokenizer
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

    # 创建数据加载器
    train_loader, val_loader = create_dataloaders(data, tokenizer, batch_size, max_length)

    # 初始化模型
    model = DistilBertMultiLabelClassifier(
        n_labels=len(data['label_names']),
        dropout_rate=dropout_rate,
        hidden_dim=hidden_dim
    )

    # 初始化训练器
    trainer = MultiLabelTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        label_names=data['label_names'],
        learning_rate=learning_rate,
        num_epochs=num_epochs,
        warmup_steps=warmup_steps,
        device=device
    )

    # 训练
    history = trainer.train(save_path=save_path)

    # 绘制训练曲线
    plot_training_history(history)

    # 显示每标签指标
    per_label = trainer.get_per_label_metrics()
    print("\n" + "="*60)
    print("每标签性能")
    print("="*60)
    print(per_label.to_string(index=False))

    return trainer

print("✓ 主训练函数定义完成")
print("\n所有组件已准备就绪! 请继续执行下一个cell开始训练。")

执行训练

In [ ]:
# 配置路径 (请根据您的数据修改)
CSV_PATH = 'dataset_A_news_full_10500.csv'  # 数据文件路径
TEXT_COL = 'title'                          # 文本列名
LABEL_COL = 'classes_str'                   # 标签列名

# 如果在Colab中，先上传文件
# from google.colab import files
# uploaded = files.upload()

# 检查文件
if not os.path.exists(CSV_PATH):
    print(f"⚠️ 文件未找到: {CSV_PATH}")
    print("请上传数据文件或修改CSV_PATH变量")
else:
    # Step 1: 加载数据
    print("Step 1: 加载数据...")
    data = load_and_prepare_data(
        csv_path=CSV_PATH,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        test_size=0.2,
        rare_threshold=10
    )

    # Step 2: 训练模型
    print("\nStep 2: 训练模型...")
    trainer = train_distilbert_model(
        data=data,
        batch_size=16,        # GPU内存不足时改为8
        learning_rate=2e-5,
        num_epochs=5,
        max_length=128,       # 可改为64加速训练
        save_path='distilbert_raise26.pth'
    )

    # Step 3: 显示最终结果
    print("\n" + "="*60)
    print("最终结果")
    print("="*60)
    print(f"最佳 Macro-F1: {trainer.best_macro_f1:.4f}")
    print(f"最终 Micro-F1: {trainer.history['val_micro_f1'][-1]:.4f}")

使用训练好的模型进行预测

In [ ]:
# 加载模型
predictor = DistilBertPredictor('distilbert_raise26.pth')

# 测试样例
test_texts = [
    "AI assistant helps workers become more productive",
    "Concerns about AI replacing human jobs in factories",
    "New AI system improves medical diagnosis accuracy",
    "AI ethics debate continues in academic circles"
]

# 预测
results = predictor.predict(test_texts, threshold=0.4, return_probs=True)

# 显示结果
print("\n预测结果:")
print("="*60)
for r in results:
    print(f"\n文本: {r['text']}")
    print(f"标签: {r['predicted_labels']}")
    if r.get('predictions'):
        print("概率:")
        for label, prob in r['predictions'][:3]:
            print(f"  - {label}: {prob:.3f}")

 对Dataset C进行预测

In [ ]:
# 配置Dataset C路径
PATH_C = 'Dataset_C_prompts_&_queries.csv'  # 修改为您的文件路径

if os.path.exists(PATH_C):
    # 加载数据
    dfC = pd.read_csv(PATH_C)
    print(f"Dataset C: {len(dfC)} 条记录")

    # 使用训练好的模型预测
    predictor = DistilBertPredictor('distilbert_raise26.pth')

    # 预测LLM输出的标签
    dfC_pred = predictor.predict_dataframe(
        df=dfC,
        text_col='LLM_output',
        threshold=0.3
    )

    # 分析各LLM的标签分布
    print("\n各LLM的标签分布:")
    print("="*60)

    for llm in dfC_pred['LLM'].unique():
        llm_df = dfC_pred[dfC_pred['LLM'] == llm]
        all_labels = [l for labels in llm_df['predicted_labels'] for l in labels]
        label_counts = pd.Series(all_labels).value_counts()

        print(f"\n{llm}:")
        print(label_counts.head(5).to_string())

    # 保存结果
    dfC_pred.to_csv('dataset_C_with_predictions.csv', index=False)
    print("\n✓ 预测结果已保存至: dataset_C_with_predictions.csv")
else:
    print(f"⚠️ Dataset C未找到: {PATH_C}")
    print("请上传文件或修改路径")

# Quant Analysis

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 1: 安装依赖 & 导入库
# ============================================================================

!pip install -q yfinance arch

import os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

from scipy import stats
from scipy.stats import pearsonr, spearmanr
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from arch import arch_model

print('✓ Quant Analysis 依赖加载完成')

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 2: 配置常量 & 行业暴露度矩阵
# ============================================================================

# AI相关股票/ETF
AI_TICKERS = {
    'NVDA': 'NVIDIA',
    'GOOGL': 'Alphabet',
    'MSFT': 'Microsoft',
    'META': 'Meta',
    'AMZN': 'Amazon',
    'AMD': 'AMD',
    'BOTZ': 'Global X Robotics & AI ETF',
    'AIQ': 'Global X AI & Technology ETF',
    'SPY': 'S&P 500 ETF',
    'QQQ': 'NASDAQ 100 ETF'
}

# 行为标签 → 行业暴露度映射 (权重0-1)
LABEL_INDUSTRY_EXPOSURE = {
    'Work, Jobs & Economy': {
        'labor': 1.0, 'automation': 0.9, 'enterprise': 0.8
    },
    'Technology & Interaction': {
        'tech': 1.0, 'semiconductors': 0.8, 'software': 0.9
    },
    'Health, Safety & Risk': {
        'healthcare': 1.0, 'biotech': 0.7, 'risk': 0.6
    },
    'Learning, Knowledge & Education': {
        'edtech': 0.9, 'enterprise': 0.5
    },
    'Society, Ethics & Culture': {
        'regulation': 0.8, 'risk': 0.7
    },
    'Cognitive & Decision-Making': {
        'enterprise': 0.7, 'fintech': 0.6
    },
    'Creativity, Expression & Identity': {
        'media': 0.8, 'entertainment': 0.7
    },
    'Social Interaction & Relationships': {
        'social_media': 0.9, 'communication': 0.7
    },
    'Emotion, Motivation & Well-being': {
        'healthcare': 0.6, 'consumer': 0.5
    },
    'Routine, Lifestyle & Behavior': {
        'consumer': 0.8, 'automation': 0.6
    },
    'Human Roles': {
        'labor': 0.9, 'enterprise': 0.7
    },
    'Sentiment (Positive / Negative Feelings)': {
        'market_sentiment': 1.0
    }
}

# Ticker → 行业映射
TICKER_INDUSTRY = {
    'NVDA': ['tech', 'semiconductors', 'automation'],
    'GOOGL': ['tech', 'software', 'enterprise', 'media'],
    'MSFT': ['tech', 'software', 'enterprise', 'automation'],
    'META': ['social_media', 'media', 'communication'],
    'AMZN': ['tech', 'enterprise', 'consumer', 'automation'],
    'AMD': ['tech', 'semiconductors'],
    'BOTZ': ['automation', 'tech'],
    'AIQ': ['tech', 'automation', 'enterprise'],
    'SPY': ['market_sentiment'],
    'QQQ': ['tech', 'market_sentiment']
}

print(f'✓ 配置加载完成')
print(f'  - {len(AI_TICKERS)} 个股票/ETF')
print(f'  - {len(LABEL_INDUSTRY_EXPOSURE)} 个行为标签')
print(f'  - {len(set(ind for inds in TICKER_INDUSTRY.values() for ind in inds))} 个行业类别')

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 3: 加载新闻数据
# ============================================================================

# 复用已加载的dfA，如果不存在则重新加载
if 'dfA' not in dir() or dfA is None:
    csv_path = '/content/dataset_A_news_full_10500.csv'
    if os.path.exists(csv_path):
        dfA = pd.read_csv(csv_path)
        dfA['date'] = pd.to_datetime(dfA['date'], errors='coerce')
        dfA = dfA.dropna(subset=['date', 'title', 'classes_str'])
    else:
        raise FileNotFoundError('请上传 dataset_A_news_full_10500.csv')

df_news = dfA.copy()
df_news['labels'] = df_news['classes_str'].apply(
    lambda s: [l.strip() for l in str(s).split(';') if l.strip()]
)

print(f'✓ 新闻数据: {len(df_news)} 条')
print(f'  日期范围: {df_news["date"].min().date()} 至 {df_news["date"].max().date()}')
print(f'\n前3条样本:')
display(df_news[['date', 'title', 'classes_str']].head(3))

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 4: 行为标签时间序列聚合
# ============================================================================

def aggregate_news_by_date(df):
    """按日期聚合新闻，生成每日标签计数和比例"""
    all_labels = sorted(set(l for labels in df['labels'] for l in labels))
    daily_data = []

    for date, group in df.groupby(df['date'].dt.date):
        row = {'date': pd.Timestamp(date), 'news_count': len(group)}

        # 统计每个标签
        label_counts = {}
        for labels in group['labels']:
            for label in labels:
                label_counts[label] = label_counts.get(label, 0) + 1

        # 绝对计数
        for label in all_labels:
            row[f'count_{label}'] = label_counts.get(label, 0)

        # 比例
        total = sum(label_counts.values())
        for label in all_labels:
            row[f'pct_{label}'] = label_counts.get(label, 0) / max(total, 1)

        # 情绪强度
        sentiment_count = label_counts.get('Sentiment (Positive / Negative Feelings)', 0)
        row['sentiment_intensity'] = sentiment_count / max(row['news_count'], 1)

        daily_data.append(row)

    daily_df = pd.DataFrame(daily_data).sort_values('date').set_index('date')
    return daily_df, all_labels

# 执行聚合
daily_news, label_list = aggregate_news_by_date(df_news)

print(f'✓ 生成 {len(daily_news)} 天的每日聚合数据')
print(f'\n标签列表 ({len(label_list)}个):')
for i, label in enumerate(label_list, 1):
    print(f'  {i:2d}. {label}')

# 可视化每日新闻量
plt.figure(figsize=(14, 4))
plt.bar(daily_news.index, daily_news['news_count'], color='steelblue', alpha=0.7)
plt.title('Daily News Volume', fontweight='bold')
plt.ylabel('Number of News')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 5: AI股票/ETF数据获取与处理
# ============================================================================
# 注意: 由于新闻日期为2025年6-9月(未来)，使用基于真实市场统计的模拟数据

def generate_market_data(tickers, news_index):
    """
    基于真实市场统计特性生成模拟数据
    参数来自2024年AI股票的实际波动率和收益特征
    """
    np.random.seed(42)

    # 各股票参数 (基于2024年真实历史数据)
    stock_params = {
        'NVDA': {'init': 500, 'drift': 0.0012, 'vol': 0.032},   # 高波动成长股
        'GOOGL': {'init': 175, 'drift': 0.0005, 'vol': 0.016},
        'MSFT': {'init': 420, 'drift': 0.0004, 'vol': 0.013},
        'META': {'init': 510, 'drift': 0.0008, 'vol': 0.022},
        'AMZN': {'init': 185, 'drift': 0.0005, 'vol': 0.018},
        'AMD': {'init': 165, 'drift': 0.0007, 'vol': 0.028},
        'BOTZ': {'init': 30, 'drift': 0.0004, 'vol': 0.020},
        'AIQ': {'init': 35, 'drift': 0.0004, 'vol': 0.018},
        'SPY': {'init': 555, 'drift': 0.0003, 'vol': 0.010},    # 低波动指数
        'QQQ': {'init': 485, 'drift': 0.0004, 'vol': 0.013}
    }

    # 扩展日期范围
    start = news_index.min() - timedelta(days=10)
    end = news_index.max() + timedelta(days=5)
    trading_days = pd.date_range(start=start, end=end, freq='B')

    prices = pd.DataFrame(index=trading_days)

    for ticker in tickers.keys():
        p = stock_params.get(ticker, {'init': 100, 'drift': 0.0003, 'vol': 0.015})
        n = len(trading_days)

        # 几何布朗运动 (GBM) 模拟
        daily_returns = np.random.normal(p['drift'], p['vol'], n)
        price_series = [p['init']]
        for r in daily_returns[:-1]:
            price_series.append(price_series[-1] * (1 + r))

        prices[ticker] = price_series

    return prices

# 生成市场数据
prices_df = generate_market_data(AI_TICKERS, daily_news.index)
returns_df = prices_df.pct_change().dropna()
volume_df = pd.DataFrame(
    np.random.randint(1000000, 50000000, size=prices_df.shape),
    index=prices_df.index, columns=prices_df.columns
)

print(f'✓ 生成 {len(prices_df)} 天的模拟市场数据')
print(f'  Tickers: {list(prices_df.columns)}')
print(f'  日期范围: {prices_df.index.min().date()} 至 {prices_df.index.max().date()}')

# 价格走势可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 归一化价格
norm_prices = prices_df / prices_df.iloc[0] * 100
for col in ['NVDA', 'GOOGL', 'MSFT', 'QQQ']:
    if col in norm_prices.columns:
        axes[0].plot(norm_prices.index, norm_prices[col], label=col, linewidth=1.5)
axes[0].set_title('Normalized Price (Base=100)', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Price Index')

# 收益率分布
returns_df[['NVDA', 'QQQ']].plot(kind='hist', bins=50, alpha=0.6, ax=axes[1])
axes[1].set_title('Return Distribution', fontweight='bold')
axes[1].set_xlabel('Daily Return')

plt.tight_layout()
plt.show()

print('\n价格数据样本:')
display(prices_df.head())

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 6: GARCH(1,1) 波动率建模
# ============================================================================

def fit_garch_model(returns, ticker_name=''):
    """
    拟合GARCH(1,1)模型估计条件波动率

    Returns:
        annualized_vol: 年化条件波动率序列
        result: 模型拟合结果
    """
    ret = returns.dropna() * 100  # 转换为百分比

    if len(ret) < 30:
        print(f'  {ticker_name}: 数据不足，使用滚动波动率')
        return returns.rolling(window=5).std() * np.sqrt(252), None

    try:
        model = arch_model(ret, vol='Garch', p=1, q=1, mean='Zero', rescale=False)
        result = model.fit(disp='off', show_warning=False)

        # 条件波动率 (转回原始尺度并年化)
        cond_vol = result.conditional_volatility / 100
        annualized_vol = cond_vol * np.sqrt(252)

        print(f'  {ticker_name}: GARCH(1,1) ✓ | 平均年化波动率: {annualized_vol.mean():.1%}')

        return annualized_vol, result

    except Exception as e:
        print(f'  {ticker_name}: GARCH失败，使用滚动波动率')
        return returns.rolling(window=5).std() * np.sqrt(252), None

print('=' * 55)
print('GARCH(1,1) 波动率建模')
print('=' * 55)

# 对主要股票拟合GARCH
vol_dict = {}
garch_results = {}

for ticker in ['NVDA', 'GOOGL', 'MSFT', 'META', 'AMD', 'QQQ', 'SPY']:
    if ticker in returns_df.columns:
        vol, result = fit_garch_model(returns_df[ticker], ticker)
        vol_dict[f'{ticker}_vol'] = vol
        if result is not None:
            garch_results[ticker] = result

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 7: 合并新闻与市场数据
# ============================================================================

# 创建市场数据汇总
market_df = pd.DataFrame(index=returns_df.index)

# 添加收益率
for col in returns_df.columns:
    market_df[f'{col}_ret'] = returns_df[col]

# 添加波动率
for col in vol_df.columns:
    market_df[col] = vol_df[col]

# 合并 (inner join 确保日期对齐)
merged_df = daily_news.join(market_df, how='inner')

print(f'✓ 合并后数据: {len(merged_df)} 天, {merged_df.shape[1]} 列')
print(f'  日期范围: {merged_df.index.min().date()} 至 {merged_df.index.max().date()}')

# 显示列分组
news_cols = [c for c in merged_df.columns if c.startswith('pct_') or c.startswith('count_')]
market_cols = [c for c in merged_df.columns if '_ret' in c or '_vol' in c]
print(f'\n  新闻特征: {len(news_cols)} 列')
print(f'  市场特征: {len(market_cols)} 列')

display(merged_df.head())

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 8: 相关性分析
# ============================================================================

def analyze_correlations(df, labels, market_cols):
    """分析新闻标签与市场指标的相关性"""
    results = []

    for label in labels:
        label_col = f'pct_{label}'
        if label_col not in df.columns:
            continue

        for market_col in market_cols:
            if market_col not in df.columns:
                continue

            valid = df[[label_col, market_col]].dropna()
            if len(valid) < 10:
                continue

            # Pearson相关
            pearson_r, pearson_p = pearsonr(valid[label_col], valid[market_col])

            # Spearman相关
            spearman_r, spearman_p = spearmanr(valid[label_col], valid[market_col])

            results.append({
                'label': label,
                'market': market_col,
                'pearson_r': pearson_r,
                'pearson_p': pearson_p,
                'spearman_r': spearman_r,
                'spearman_p': spearman_p,
                'n_obs': len(valid),
                'significant': pearson_p < 0.05
            })

    return pd.DataFrame(results)

# 执行相关性分析
market_cols = [c for c in merged_df.columns if '_vol' in c or '_ret' in c]
corr_df = analyze_correlations(merged_df, label_list, market_cols)
corr_df['abs_r'] = corr_df['pearson_r'].abs()
corr_df = corr_df.sort_values('abs_r', ascending=False)

print('=' * 70)
print('相关性分析: 新闻标签 × 市场指标')
print('=' * 70)

# Top显著相关
print('\n📊 Top 15 显著相关性 (p < 0.05):')
print('-' * 70)
sig_corr = corr_df[corr_df['significant']].head(15)
for _, row in sig_corr.iterrows():
    sign = '📈' if row['pearson_r'] > 0 else '📉'
    print(f"  {sign} {row['label'][:30]:<32} × {row['market']:<15} r={row['pearson_r']:+.3f} (p={row['pearson_p']:.4f})")

print(f'\n✓ 总检验对数: {len(corr_df)}')
print(f'✓ 显著相关数: {corr_df["significant"].sum()} ({corr_df["significant"].mean()*100:.1f}%)')

# 相关性热图
plt.figure(figsize=(14, 8))
pivot_corr = corr_df.pivot_table(index='label', columns='market', values='pearson_r', aggfunc='first')

# 只显示top标签
top_labels = corr_df.groupby('label')['abs_r'].max().nlargest(10).index
pivot_corr = pivot_corr.loc[pivot_corr.index.isin(top_labels)]

sns.heatmap(pivot_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Correlation'}, linewidths=0.5)
plt.title('Label-Market Correlation Heatmap (Top 10 Labels)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 9: Granger因果检验 (新闻 → 市场波动)
# ============================================================================

def granger_causality_test(df, labels, target='AI_vol_index', max_lag=5):
    """
    检验新闻标签是否Granger-cause市场波动率
    H0: 新闻不能预测波动率
    """
    results = []

    for label in labels:
        label_col = f'pct_{label}'
        if label_col not in df.columns:
            continue

        # 准备数据
        test_data = df[[label_col, target]].dropna()

        if len(test_data) < max_lag * 4:
            continue

        try:
            # Granger检验 (注意顺序: [y, x] 检验 x -> y)
            gc_result = grangercausalitytests(
                test_data[[target, label_col]],
                maxlag=max_lag,
                verbose=False
            )

            # 提取各滞后阶的p值
            p_values = {lag: gc_result[lag][0]['ssr_ftest'][1] for lag in range(1, max_lag+1)}
            min_p = min(p_values.values())
            best_lag = [k for k, v in p_values.items() if v == min_p][0]

            results.append({
                'label': label,
                'target': target,
                'best_lag': best_lag,
                'min_p_value': min_p,
                'p_lag1': p_values.get(1),
                'p_lag2': p_values.get(2),
                'p_lag3': p_values.get(3),
                'significant': min_p < 0.05,
                'n_obs': len(test_data)
            })

        except Exception as e:
            continue

    return pd.DataFrame(results)

# 执行Granger检验
print('=' * 65)
print('Granger因果检验: 新闻标签 → AI波动率指数')
print('=' * 65)

granger_df = granger_causality_test(merged_df, label_list, 'AI_vol_index', max_lag=5)
granger_df = granger_df.sort_values('min_p_value')

print(f"\n{'标签':<35} {'最佳滞后':<8} {'p值':<12} {'显著'}")
print('-' * 65)

for _, row in granger_df.iterrows():
    sig_mark = '***' if row['min_p_value'] < 0.01 else ('**' if row['min_p_value'] < 0.05 else ('*' if row['min_p_value'] < 0.1 else ''))
    print(f"{row['label'][:33]:<35} {row['best_lag']:<8} {row['min_p_value']:.4f}       {sig_mark}")

sig_count = granger_df['significant'].sum()
print(f'\n✓ {sig_count}/{len(granger_df)} 个标签显示显著Granger因果关系 (p<0.05)')

# 可视化
plt.figure(figsize=(12, 8))
gs = granger_df.sort_values('min_p_value')
colors = ['green' if p < 0.05 else ('orange' if p < 0.1 else 'gray') for p in gs['min_p_value']]

plt.barh(range(len(gs)), gs['min_p_value'], color=colors, alpha=0.7)
plt.axvline(x=0.05, color='red', linestyle='--', linewidth=2, label='p=0.05')
plt.axvline(x=0.10, color='orange', linestyle=':', linewidth=1.5, label='p=0.10')
plt.yticks(range(len(gs)), [l[:30] for l in gs['label']], fontsize=9)
plt.xlabel('Granger Causality p-value')
plt.title('Granger Causality Test: News Labels → Market Volatility', fontweight='bold')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 10: 行业暴露度分析
# ============================================================================

def compute_industry_exposure(df, labels, exposure_map):
    """
    基于新闻标签计算每日行业暴露度得分
    """
    # 收集所有行业
    all_industries = sorted(set(
        ind for label_exp in exposure_map.values() for ind in label_exp.keys()
    ))

    exposure_scores = []

    for idx in df.index:
        row = {'date': idx}

        for industry in all_industries:
            score = 0
            for label in labels:
                label_col = f'pct_{label}'
                if label_col not in df.columns:
                    continue

                # 标签比例 × 行业暴露权重
                label_pct = df.loc[idx, label_col]
                weight = exposure_map.get(label, {}).get(industry, 0)
                score += label_pct * weight

            row[f'exp_{industry}'] = score

        exposure_scores.append(row)

    exposure_df = pd.DataFrame(exposure_scores).set_index('date')
    return exposure_df, all_industries

# 计算行业暴露度
exposure_df, industries = compute_industry_exposure(merged_df, label_list, LABEL_INDUSTRY_EXPOSURE)

print('=' * 55)
print('行业暴露度分析')
print('=' * 55)

print(f'\n行业类别 ({len(industries)}个): {industries}')

print('\n每日平均行业暴露度:')
print('-' * 40)
means = exposure_df.mean().sort_values(ascending=False)
for col in means.index:
    industry = col.replace('exp_', '')
    print(f'  {industry:<20}: {means[col]:.4f}')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 时间序列
ax1 = axes[0]
main_industries = ['tech', 'automation', 'labor', 'market_sentiment', 'enterprise']
for ind in main_industries:
    col = f'exp_{ind}'
    if col in exposure_df.columns:
        ax1.plot(exposure_df.index, exposure_df[col], label=ind, linewidth=1.5)
ax1.set_title('Industry Exposure Over Time', fontweight='bold')
ax1.set_ylabel('Exposure Score')
ax1.legend(loc='upper right')

# 相关性: 行业暴露 vs 市场
ax2 = axes[1]
exp_market_corr = []
for col in exposure_df.columns:
    if 'AI_vol_index' in merged_df.columns:
        valid = pd.concat([exposure_df[col], merged_df['AI_vol_index']], axis=1).dropna()
        if len(valid) > 10:
            r, p = pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
            exp_market_corr.append({'industry': col.replace('exp_', ''), 'corr': r, 'p': p})

exp_corr_df = pd.DataFrame(exp_market_corr).sort_values('corr')
colors = ['green' if abs(c) > 0.1 else 'gray' for c in exp_corr_df['corr']]
ax2.barh(exp_corr_df['industry'], exp_corr_df['corr'], color=colors, alpha=0.7)
ax2.axvline(0, color='black', linewidth=0.5)
ax2.set_title('Industry Exposure vs Market Volatility', fontweight='bold')
ax2.set_xlabel('Correlation')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# QUANT ANALYSIS CELL 11: 综合可视化仪表板
# ============================================================================

fig = plt.figure(figsize=(20, 14))

# --- Panel 1: 新闻量 vs 市场波动率 ---
ax1 = plt.subplot(2, 3, 1)
ax1.bar(merged_df.index, merged_df['news_count'], alpha=0.5, color='steelblue', label='News Count')
ax1.set_ylabel('Daily News Count', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1_twin = ax1.twinx()
ax1_twin.plot(merged_df.index, merged_df['AI_vol_index'], color='crimson', linewidth=2, label='AI Volatility')
ax1_twin.set_ylabel('AI Volatility Index', color='crimson')
ax1_twin.tick_params(axis='y', labelcolor='crimson')
ax1.set_title('News Volume vs Market Volatility', fontsize=12, fontweight='bold')

# --- Panel 2: 相关性热图 ---
ax2 = plt.subplot(2, 3, 2)
pivot = corr_df.pivot_table(index='label', columns='market', values='pearson_r', aggfunc='first')
top_labels = corr_df.groupby('label')['abs_r'].max().nlargest(8).index
vol_cols = [c for c in pivot.columns if '_vol' in c]
pivot_sub = pivot.loc[pivot.index.isin(top_labels), vol_cols]
sns.heatmap(pivot_sub, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax2)
ax2.set_title('Label-Volatility Correlation', fontsize=12, fontweight='bold')

# --- Panel 3: Granger因果 ---
ax3 = plt.subplot(2, 3, 3)
gs = granger_df.sort_values('min_p_value')
colors = ['green' if p < 0.05 else 'gray' for p in gs['min_p_value']]
ax3.barh(range(len(gs)), gs['min_p_value'], color=colors, alpha=0.7)
ax3.axvline(x=0.05, color='red', linestyle='--', linewidth=2)
ax3.set_yticks(range(len(gs)))
ax3.set_yticklabels([l[:25] for l in gs['label']], fontsize=9)
ax3.set_xlabel('p-value')
ax3.set_title('Granger Causality: News → Vol', fontsize=12, fontweight='bold')
ax3.invert_yaxis()

# --- Panel 4: 每日标签分布 ---
ax4 = plt.subplot(2, 3, 4)
top5_labels = ['Work, Jobs & Economy', 'Technology & Interaction',
               'Learning, Knowledge & Education', 'Health, Safety & Risk',
               'Society, Ethics & Culture']
label_data = [merged_df[f'pct_{l}'].values for l in top5_labels if f'pct_{l}' in merged_df.columns]
ax4.stackplot(merged_df.index, label_data, labels=top5_labels[:len(label_data)], alpha=0.7)
ax4.set_title('Daily Label Distribution (Top 5)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Proportion')
ax4.legend(loc='upper left', fontsize=8)

# --- Panel 5: 行业暴露度趋势 ---
ax5 = plt.subplot(2, 3, 5)
for ind in ['tech', 'automation', 'labor']:
    col = f'exp_{ind}'
    if col in exposure_df.columns:
        ax5.plot(exposure_df.index, exposure_df[col], label=ind, linewidth=2)
ax5.set_title('Industry Exposure Trends', fontsize=12, fontweight='bold')
ax5.set_ylabel('Exposure Score')
ax5.legend()

# --- Panel 6: 股票收益相关性 ---
ax6 = plt.subplot(2, 3, 6)
ret_cols = [c for c in merged_df.columns if '_ret' in c and c.replace('_ret', '') in ['NVDA', 'GOOGL', 'MSFT', 'QQQ']]
if len(ret_cols) > 0:
    ret_corr = merged_df[ret_cols].corr()
    sns.heatmap(ret_corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax6)
    ax6.set_title('Stock Return Correlations', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('quant_analysis_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ 仪表板已保存: quant_analysis_dashboard.png')